# JJ100-139

In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

## Merge JJ110 and JJ100-109+JJ110-118

In [7]:


# Chargement des deux CSV
df1 = pd.read_csv("../List-of-zones/_old_csv_exports/Himanis_Seg_Actes_JJ100-109_JJ111-118_labelstudio.csv", sep=",")  # adapter le séparateur si besoin
df2 = pd.read_csv("../List-of-zones/_old_csv_exports/Himanis_Seg_Actes_JJ110_labelstudio.csv", sep=",")

# Concaténation
df = pd.concat([df1, df2], ignore_index=True)

# Suppression des lignes vides (lignes sans image)
df = df.dropna(subset=["image"])
df = df[df["image"].str.strip() != ""]

# Vérification des doublons sur "image"
doublons = df[df.duplicated(subset=["image"], keep=False)]
if not doublons.empty:
    print(f"⚠️  {doublons['image'].nunique()} image(s) en doublon détectée(s) :")
    print(doublons[["annotation_id", "id", "image", "label"]].to_string(index=False))
    # Suppression des doublons, on garde la première occurrence
else:
    print("✅ Aucun doublon détecté sur le champ 'image'.")

def merge_duplicates(group):
    if len(group) == 1:
        return group.iloc[0]
    first = group.iloc[0].copy()
    merged_label = first["label"]
    for _, row in group.iloc[1:].iterrows():
        merged_label = str(merged_label)[:-1] + str(row["label"])[1:]
    first["label"] = merged_label
    return first

df = df.groupby("image", sort=False).apply(merge_duplicates).reset_index(drop=True)


# Renumérotation de 1 à N
df = df.reset_index(drop=True)
df["annotation_id"] = df.index + 1
df["id"] = df.index + 1

print(f"\n✅ Dataset final : {len(df)} lignes")
print(df.head())

# Export optionnel
df.to_csv("../List-of-zones/_old_csv_exports/Himanis_Seg_Actes_JJ100-118_labelstudio.csv", sep=",", index=False)

⚠️  63 image(s) en doublon détectée(s) :
 annotation_id   id                                                                           image                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

C:\Users\stutzmann\AppData\Local\Temp\ipykernel_31064\3064658134.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("image", sort=False).apply(merge_duplicates).reset_index(drop=True)


# Add image info

In [9]:
# Pour autres fichiers que JJ100-118
# df = pd.read_csv("../List-of-zones/_old_csv_exports/Himanis_Seg_Actes_JJ119-132_labelstudio.csv", sep=",")
# df = pd.read_csv("../List-of-zones/_old_csv_exports/Himanis_Seg_Actes_JJ133-139_labelstudio.csv", sep=",")


# Chargement du fichier image_data
image_data_path ="../List-of-zones/image_data/JJ100-JJ118_image_data.csv"
# image_data_path = "../List-of-zones/image_data/JJ119-JJ132_image_data.csv"
# image_data_path = "../List-of-zones/image_data/JJ133-JJ139_image_data.csv"


df_image_data = pd.read_csv(image_data_path, sep=",")  # adapter le séparateur si besoin

# Merge sur urlResizedImage <-> image
df = df.merge(
    df_image_data[["urlResizedImage", "urlImage", "folderPath", "imageLabel", "imageFileName", "imageWidthAsDownloaded", "imageHeightAsDownloaded"]],
    left_on="image",
    right_on="urlResizedImage",
    how="left"
).drop(columns=["urlResizedImage"])

# Vérification des lignes non matchées
unmatched = df[df["urlImage"].isna()]
if not unmatched.empty:
    print(f"⚠️  {len(unmatched)} ligne(s) sans correspondance dans image_data :")
    print(unmatched[["annotation_id", "image"]].to_string(index=False))
else:
    print("✅ Toutes les images ont été matchées.")

✅ Toutes les images ont été matchées.


# Create json import files

In [11]:
import json


output_path = "../List-of-zones/label_studio_import/label_studio_import_JJ100-118.json"



# Extraction registre + numéro d'ordre depuis imageFileName
# ex: Paris_Archives_Nationales_JJ096_100.jpg → registre=JJ096, ordre=100
def parse_image_stem(filename):
    stem = str(filename).rsplit(".", 1)[0]
    parts = stem.split("_")
    registre = parts[-2]
    ordre = int(parts[-1])
    return registre, ordre

df[["Registre", "Ordre"]] = df["imageFileName"].apply(
    lambda p: pd.Series(parse_image_stem(p))
)


# Images présentes dans image_data mais absentes du CSV d'annotations
images_annotees = set(df["image"])
df_manquantes = df_image_data[~df_image_data["urlResizedImage"].isin(images_annotees)].copy()
df_manquantes[["Registre", "Ordre"]] = df_manquantes["imageFileName"].apply(
    lambda p: pd.Series(parse_image_stem(p))
)
print(f"ℹ️  {len(df_manquantes)} image(s) sans annotation → tasks vides ajoutées")
if not df_manquantes.empty:
    print(df_manquantes[["Registre", "Ordre", "imageFileName"]].to_string(index=False))

# Tri par registre puis numéro d'ordre
df = df.sort_values(["Registre", "Ordre"]).reset_index(drop=True)

tasks = []

for _, row in df.iterrows():
    img_w = int(row["imageWidthAsDownloaded"])
    img_h = int(row["imageHeightAsDownloaded"])

    # Parser le label JSON
    try:
        regions = json.loads(row["label"]) if pd.notna(row["label"]) and row["label"] != "" else []
    except Exception:
        regions = []

    # Construire les annotations au format Label Studio
    annotations = []
    for i, r in enumerate(regions):
        annotations.append({
            "id": f"region_{i}",
            "type": "rectanglelabels",
            "value": {
                "x": r["x"],
                "y": r["y"],
                "width": r["width"],
                "height": r["height"],
                "rotation": r.get("rotation", 0),
                "rectanglelabels": r.get("rectanglelabels", [])
            },
            "to_name": "image",
            "from_name": "label",
            "image_rotation": 0,
            "original_width": img_w,
            "original_height": img_h
        })

    task = {
        "data": {
            "image": row["image"],
            "image_path": row["urlImage"],
            "registre": row["Registre"],
            "ordre": int(row["Ordre"]),
        },
        "annotations": [{"result": annotations}],
        "meta": {
            "source_file": row["imageFileName"],
            "folder": row["folderPath"],
            "image_label": row["imageLabel"]
        }
    }
    tasks.append(task)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print(f"✓ {len(tasks)} tâche(s) générée(s) → {output_path}")

ℹ️  20 image(s) sans annotation → tasks vides ajoutées
Registre  Ordre                                                                                                  imageFileName
   JJ101      1   images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ101\Paris_Archives_Nationales_JJ101_1.jpg
   JJ102      1   images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ102\Paris_Archives_Nationales_JJ102_1.jpg
   JJ102    173 images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ102\Paris_Archives_Nationales_JJ102_173.jpg
   JJ103      1   images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ103\Paris_Archives_Nationales_JJ103_1.jpg
   JJ104      1   images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ104\Paris_Archives_Nationales_JJ104_1.jpg
   JJ105      1   images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ105\Paris_Archives_Nationales_JJ105_1.jpg
   JJ106      1   images_registres_AN_JJ035_JJ211/images